In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import glob

# ===================== 你已有的特征文件夹 =====================
MAE_FEAT_DIR = "/root/autodl-tmp/mae_features"
VIT_FEAT_DIR = "/root/autodl-tmp/vit_features"
FEAT_DIM = 768
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =============================================================================

# ===================== 数据集 =====================
class FeatureDataset(Dataset):
    def __init__(self, feat_dir):
        self.files = sorted(glob.glob(os.path.join(feat_dir, "*.npy")))
        print(f"✅ 加载特征数量：{len(self.files)}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        x = np.load(self.files[idx])
        return torch.tensor(x, dtype=torch.float32)

# ===================== 特征质量评估模型（自监督对比）=====================
class FeatureReconstructor(nn.Module):
    def __init__(self, dim=768):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )
        self.decoder = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon, z

# ===================== 评估特征：重构误差越小 → 特征越好 =====================
def evaluate(feat_dir, name):
    print("\n=====================================")
    print(f"🚀 评估：{name}")
    print("=====================================")

    dataset = FeatureDataset(feat_dir)
    loader = DataLoader(dataset, batch_size=128, shuffle=True)

    model = FeatureReconstructor(FEAT_DIM).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    total_loss = 0.0
    model.train()
    for step, x in enumerate(loader):
        x = x.to(DEVICE)
        optimizer.zero_grad()
        recon, _ = model(x)
        loss = criterion(recon, x)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    final_loss = total_loss / len(loader)
    print(f"✅ {name} 重构损失：{final_loss:.6f}")
    return final_loss

# ===================== 主程序 =====================
if __name__ == "__main__":
    print("🔥 真实特征公平对比（使用你已有的特征）🔥")
    print("🔥 损失越小 → 特征质量越高 🔥")

    loss_mae = evaluate(MAE_FEAT_DIR, "MAE 自监督特征")
    loss_vit = evaluate(VIT_FEAT_DIR, "纯 ViT 特征")

    print("\n" + "="*60)
    print("📊 最终真实对比结果")
    print("="*60)
    print(f"MAE 特征  重构损失：{loss_mae:.6f}")
    print(f"纯ViT 特征 重构损失：{loss_vit:.6f}")
    print("="*60)

    if loss_mae < loss_vit:
        print("🏆 结论：MAE 特征质量 **更强**")
    else:
        print("🏆 结论：纯 ViT 特征质量 **更强**")